# Recurrent Neural Networks and LSTMs for Time Series

Standard feedforward networks (MLPs) treat each input window as a flat
vector of independent features.  They have no *memory* -- each prediction
is made in isolation, and patterns at one position in the window must be
re-learned at every other position.

**Recurrent Neural Networks (RNNs)** solve this by processing sequences
one step at a time, maintaining a **hidden state** that carries
information forward.  **Long Short-Term Memory (LSTM)** networks extend
this idea with a gating mechanism that can selectively remember or
forget information over long time spans.

**Contents:**
1. Why standard NNs fail for sequences
2. RNN: hidden state and the vanishing gradient problem
3. LSTM: gates and cell state
4. GRU: simplified 2-gate variant
5. Data preparation with scaling
6. Simple LSTM for airline passengers
7. Stacked LSTMs
8. Bidirectional LSTMs
9. Early stopping
10. Recursive multi-step forecasting
11. Evaluation and comparison

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras import layers

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

np.random.seed(42)
keras.utils.set_random_seed(42)

print(f"Keras version: {keras.__version__}")

---
## 1. Why Standard NNs Fail for Sequences

An MLP with a window of size $w$ treats the input as an *unordered*
set of $w$ features.  This creates three problems:

1. **No parameter sharing across time**: a spike at position 1 and a
   spike at position 5 use completely different weights.  The MLP must
   learn the same pattern independently at every position.

2. **Fixed-length context**: the window size is fixed at training time.
   If important patterns span longer than $w$ steps, the MLP cannot
   detect them.

3. **No temporal structure**: the MLP does not know that position $t-1$
   is *adjacent* to position $t$.  It must learn this from data,
   wasting capacity.

**Recurrent networks** solve all three problems by processing the
sequence one step at a time with *shared* weights and a *hidden state*
that accumulates information.

---
## 2. The Simple RNN

At each time step $t$, a simple (vanilla) RNN computes:

$$
h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b)
$$

where:
- $x_t$ is the input at time $t$
- $h_{t-1}$ is the hidden state from the previous step
- $W_{hh}$ and $W_{xh}$ are weight matrices (*shared across all time steps*)
- $b$ is the bias vector
- $\tanh$ squashes the output to $(-1, 1)$

The hidden state $h_t$ is the network's "memory" -- it carries
information about all previous inputs.  The final hidden state
$h_T$ is used for prediction.

In [ ]:
# Visualize how a simple RNN processes a sequence step by step
sequence = np.array([0.2, 0.8, 0.4, 0.9, 0.1, 0.6])

# Simulate with random weights (just for visualization)
np.random.seed(0)
W_xh = np.random.randn(1, 1) * 0.5
W_hh = np.random.randn(1, 1) * 0.5
b = np.zeros(1)

h = np.zeros(1)  # initial hidden state
hidden_states = [h.copy()]

for t, x_t in enumerate(sequence):
    h = np.tanh(W_hh @ h + W_xh @ np.array([x_t]) + b)
    hidden_states.append(h.copy())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].stem(range(len(sequence)), sequence, basefmt="grey")
axes[0].set_title("Input Sequence $x_t$")
axes[0].set_xlabel("Time step $t$")
axes[0].set_ylabel("$x_t$")

axes[1].plot(range(len(hidden_states)), [h[0] for h in hidden_states],
             marker="o", color="tab:orange")
axes[1].set_title("Hidden State $h_t$")
axes[1].set_xlabel("Time step $t$")
axes[1].set_ylabel("$h_t$")
axes[1].axhline(0, color="grey", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

print("The hidden state accumulates information from all previous inputs.")
print("The SAME weights W_xh and W_hh are used at every time step.")

### The Vanishing Gradient Problem

During backpropagation through time (BPTT), gradients are multiplied by
$W_{hh}$ at each time step.  After $T$ steps, the gradient includes a
factor of $W_{hh}^T$:

$$
\frac{\partial L}{\partial h_0} = \frac{\partial L}{\partial h_T} \cdot \prod_{t=1}^{T} \frac{\partial h_t}{\partial h_{t-1}}
$$

- If $\|W_{hh}\| < 1$: gradients **shrink exponentially** $\to$ the
  network cannot learn long-range dependencies (**vanishing gradients**)
- If $\|W_{hh}\| > 1$: gradients **explode** $\to$ training becomes
  numerically unstable (**exploding gradients**)

This is why simple RNNs struggle with sequences longer than ~10-20 steps.

In [ ]:
# Demonstrate vanishing/exploding gradients
T = 50
gradient_norms = {}

for w_val, label in [(0.8, "|W|=0.8 (vanishing)"),
                      (1.0, "|W|=1.0 (stable)"),
                      (1.2, "|W|=1.2 (exploding)")]:
    norms = [w_val ** t for t in range(T)]
    gradient_norms[label] = norms

fig, ax = plt.subplots(figsize=(10, 5))
for label, norms in gradient_norms.items():
    ax.plot(norms, label=label, linewidth=2)
ax.set_xlabel("Time steps back")
ax.set_ylabel("Gradient magnitude ($|W|^t$)")
ax.set_title("Vanishing and Exploding Gradients in Simple RNNs")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Even a small deviation from 1.0 causes exponential growth/decay.")
print("This is why simple RNNs cannot learn long-range dependencies.")

---
## 3. LSTM: Long Short-Term Memory

LSTMs (Hochreiter & Schmidhuber, 1997) solve the vanishing gradient
problem by introducing a **cell state** $C_t$ -- a "memory highway" that
runs through the entire sequence with *additive* updates (no
multiplicative decay).

Three **gates** control the flow of information:

### Forget Gate -- what to *remove* from memory

$$
f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)
$$

### Input Gate -- what to *add* to memory

$$
i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)
$$
$$
\tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C)
$$

### Cell State Update

$$
C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t
$$

### Output Gate -- what to *output* from memory

$$
o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)
$$
$$
h_t = o_t \odot \tanh(C_t)
$$

where $\sigma$ is the sigmoid function, $\odot$ is element-wise
multiplication, and $[h_{t-1}, x_t]$ denotes concatenation.

**Key insight**: the cell state $C_t$ is updated *additively*, so
gradients can flow unchanged across many time steps.  The gates
learn *what* to remember and what to forget.

In [ ]:
# Visualize LSTM gate operations on a toy example
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

T = 20
t_vals = np.arange(T)

# Simulate gate values for visualization
np.random.seed(7)
forget_gate = 1 / (1 + np.exp(-np.random.randn(T) * 2))
input_gate = 1 / (1 + np.exp(-np.random.randn(T) * 2))
output_gate = 1 / (1 + np.exp(-np.random.randn(T) * 2))
candidate = np.tanh(np.random.randn(T))

# Compute cell state
cell_state = np.zeros(T)
for t in range(1, T):
    cell_state[t] = forget_gate[t] * cell_state[t - 1] + input_gate[t] * candidate[t]

axes[0, 0].bar(t_vals, forget_gate, color="tab:red", alpha=0.7)
axes[0, 0].set_title("Forget Gate $f_t$ (what to erase)")
axes[0, 0].set_ylim(0, 1)

axes[0, 1].bar(t_vals, input_gate, color="tab:green", alpha=0.7)
axes[0, 1].set_title("Input Gate $i_t$ (what to write)")
axes[0, 1].set_ylim(0, 1)

axes[1, 0].bar(t_vals, output_gate, color="tab:blue", alpha=0.7)
axes[1, 0].set_title("Output Gate $o_t$ (what to expose)")
axes[1, 0].set_ylim(0, 1)

axes[1, 1].plot(t_vals, cell_state, color="tab:purple", marker="o", markersize=4)
axes[1, 1].set_title("Cell State $C_t$ (accumulated memory)")
axes[1, 1].axhline(0, color="grey", linestyle="--", alpha=0.4)

for ax in axes.flat:
    ax.set_xlabel("Time step $t$")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Gates are sigmoid outputs in [0, 1] -- they control information flow.")
print("Cell state can grow or shrink based on what the gates decide.")

---
## 4. GRU: Gated Recurrent Unit

The **GRU** (Cho et al., 2014) simplifies the LSTM from three gates
to two, and merges the cell state and hidden state:

### Update Gate -- how much of the old state to keep

$$
z_t = \sigma(W_z [h_{t-1}, x_t] + b_z)
$$

### Reset Gate -- how much of the old state to use for the candidate

$$
r_t = \sigma(W_r [h_{t-1}, x_t] + b_r)
$$

### Candidate and State Update

$$
\tilde{h}_t = \tanh(W [r_t \odot h_{t-1}, x_t] + b)
$$
$$
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
$$

| | LSTM | GRU |
|---|------|-----|
| Gates | 3 (forget, input, output) | 2 (update, reset) |
| State | Cell state $C_t$ + hidden state $h_t$ | Hidden state $h_t$ only |
| Parameters | More | Fewer (~25% less) |
| Performance | Slightly better on long sequences | Comparable, faster to train |

In [ ]:
# Compare parameter counts: LSTM vs GRU vs SimpleRNN
input_dim = 1
units = 64

rnn_layer = layers.SimpleRNN(units, input_shape=(10, input_dim))
lstm_layer = layers.LSTM(units, input_shape=(10, input_dim))
gru_layer = layers.GRU(units, input_shape=(10, input_dim))

# Build each by passing dummy input
dummy_input = np.zeros((1, 10, input_dim))
_ = rnn_layer(dummy_input)
_ = lstm_layer(dummy_input)
_ = gru_layer(dummy_input)

print(f"{'Layer':<15s} {'Parameters':>12s}")
print("-" * 28)
print(f"{'SimpleRNN':<15s} {rnn_layer.count_params():>12,d}")
print(f"{'GRU':<15s} {gru_layer.count_params():>12,d}")
print(f"{'LSTM':<15s} {lstm_layer.count_params():>12,d}")
print()
print(f"LSTM has ~4x the parameters of SimpleRNN (3 gates + candidate).")
print(f"GRU has ~3x the parameters of SimpleRNN (2 gates + candidate).")

---
## 5. Data Preparation

We use the **airline passengers** dataset -- 144 monthly observations
with a clear upward trend and multiplicative seasonality.  This is
a classic benchmark for sequence models.

In [ ]:
airline = pd.read_csv(DATA_DIR / "airline_passengers.csv")
airline.columns = ["Month", "Passengers"]
airline["Month"] = pd.to_datetime(airline["Month"])
airline = airline.set_index("Month")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(airline.index, airline["Passengers"], marker="o", markersize=3)
ax.set_title("Airline Passengers (1949-1960)")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of Passengers")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Shape: {airline.shape}")
print(f"Range: {airline['Passengers'].min()} to {airline['Passengers'].max()}")

In [ ]:
# Scale data to [0, 1] using MinMaxScaler
scaler = MinMaxScaler()
values = airline["Passengers"].values.reshape(-1, 1).astype("float32")
scaled = scaler.fit_transform(values).flatten()

print(f"Scaled range: [{scaled.min():.4f}, {scaled.max():.4f}]")
print(f"Scaler parameters: min={scaler.data_min_[0]}, max={scaler.data_max_[0]}")

In [ ]:
def create_sequences(data, window_size):
    """Create (X, y) pairs from a 1-D time series using a sliding window."""
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)

In [ ]:
# Create sequences -- use 12 months as window (one full seasonal cycle)
WINDOW = 12

X, y = create_sequences(scaled, WINDOW)

# LSTM expects 3D input: (samples, timesteps, features)
X = X.reshape(-1, WINDOW, 1)

# Train/test split: last 24 months for testing
n_test = 24
X_train, X_test = X[:-n_test], X[-n_test:]
y_train, y_test = y[:-n_test], y[-n_test:]

print(f"X_train shape: {X_train.shape}  (samples, timesteps, features)")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

---
## 6. Simple LSTM Model

Our first LSTM model: a single LSTM layer followed by a Dense output.

In [ ]:
# Build a simple LSTM
model_lstm = keras.Sequential([
    layers.LSTM(64, input_shape=(WINDOW, 1)),
    layers.Dense(1),
])

model_lstm.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_lstm.summary()

In [ ]:
# Train the LSTM
history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    verbose=0,
)

print(f"Final train loss: {history_lstm.history['loss'][-1]:.6f}")
print(f"Final val loss:   {history_lstm.history['val_loss'][-1]:.6f}")

In [ ]:
# Learning curves
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_lstm.history["loss"], label="Train")
ax.plot(history_lstm.history["val_loss"], label="Validation")
ax.set_title("LSTM Learning Curves")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
y_pred_scaled = model_lstm.predict(X_test, verbose=0).flatten()

# Inverse transform
y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_actual = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

mae_lstm = mean_absolute_error(y_actual, y_pred)
rmse_lstm = np.sqrt(mean_squared_error(y_actual, y_pred))

print(f"Simple LSTM -- Test Metrics")
print(f"  MAE:  {mae_lstm:.2f}")
print(f"  RMSE: {rmse_lstm:.2f}")

In [ ]:
# Plot predictions
test_dates = airline.index[-n_test:]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(airline.index, airline["Passengers"], color="tab:blue", alpha=0.5, label="Full series")
ax.plot(test_dates, y_actual, color="tab:green", marker="o", markersize=4, label="Test actual")
ax.plot(test_dates, y_pred, color="tab:red", linestyle="--", marker="s",
        markersize=4, label=f"LSTM forecast (MAE={mae_lstm:.1f})")
ax.axvline(test_dates[0], color="grey", linestyle=":", alpha=0.6)
ax.set_title("Simple LSTM Forecast -- Airline Passengers")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Stacked LSTMs

Stacking multiple LSTM layers creates a deeper model that can learn
hierarchical temporal features.  The key is `return_sequences=True`
on all layers except the last LSTM:

- `return_sequences=True`: output the hidden state at *every* time step
  (shape: `(batch, timesteps, units)`) -- needed for the next LSTM layer
- `return_sequences=False` (default): output only the *last* hidden state
  (shape: `(batch, units)`) -- feeds into Dense for prediction

In [ ]:
# Build a stacked LSTM
model_stacked = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(WINDOW, 1)),
    layers.LSTM(32),
    layers.Dense(1),
])

model_stacked.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_stacked.summary()

In [ ]:
# Train the stacked LSTM
history_stacked = model_stacked.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    verbose=0,
)

# Evaluate
y_pred_stacked_s = model_stacked.predict(X_test, verbose=0).flatten()
y_pred_stacked = scaler.inverse_transform(y_pred_stacked_s.reshape(-1, 1)).flatten()

mae_stacked = mean_absolute_error(y_actual, y_pred_stacked)
rmse_stacked = np.sqrt(mean_squared_error(y_actual, y_pred_stacked))

print(f"Stacked LSTM -- Test Metrics")
print(f"  MAE:  {mae_stacked:.2f}")
print(f"  RMSE: {rmse_stacked:.2f}")

---
## 8. Bidirectional LSTMs

A **bidirectional** LSTM processes the sequence in *both* directions
(forward and backward) and concatenates the hidden states.  This gives
each time step access to both past and future context.

For time series *forecasting* (where we never have future data at
prediction time), bidirectional LSTMs are only useful within the
*input window* -- they can better understand the context of the
historical observations, even though the forecast itself is still
forward-looking.

In [ ]:
# Build a bidirectional LSTM
model_bidir = keras.Sequential([
    layers.Bidirectional(layers.LSTM(64), input_shape=(WINDOW, 1)),
    layers.Dense(1),
])

model_bidir.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_bidir.summary()

In [ ]:
# Train the bidirectional LSTM
history_bidir = model_bidir.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    verbose=0,
)

# Evaluate
y_pred_bidir_s = model_bidir.predict(X_test, verbose=0).flatten()
y_pred_bidir = scaler.inverse_transform(y_pred_bidir_s.reshape(-1, 1)).flatten()

mae_bidir = mean_absolute_error(y_actual, y_pred_bidir)
rmse_bidir = np.sqrt(mean_squared_error(y_actual, y_pred_bidir))

print(f"Bidirectional LSTM -- Test Metrics")
print(f"  MAE:  {mae_bidir:.2f}")
print(f"  RMSE: {rmse_bidir:.2f}")

---
## 9. Early Stopping

Training for too many epochs risks **overfitting** -- the model memorizes
the training data and performs poorly on new data.  **Early stopping**
monitors the validation loss and stops training when it stops improving.

Key parameters:
- `patience`: how many epochs to wait for improvement before stopping
- `restore_best_weights`: revert to the best model seen during training

In [ ]:
# Build a fresh LSTM with early stopping
model_es = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(WINDOW, 1)),
    layers.LSTM(32),
    layers.Dense(1),
])

model_es.compile(optimizer="adam", loss="mse", metrics=["mae"])

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
    verbose=1,
)

history_es = model_es.fit(
    X_train, y_train,
    epochs=300,  # set high -- early stopping will halt training
    batch_size=16,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=0,
)

print(f"Training stopped at epoch {len(history_es.history['loss'])}")
print(f"Best val loss: {min(history_es.history['val_loss']):.6f}")

In [ ]:
# Learning curves with early stopping
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_es.history["loss"], label="Train")
ax.plot(history_es.history["val_loss"], label="Validation")
best_epoch = np.argmin(history_es.history["val_loss"])
ax.axvline(best_epoch, color="red", linestyle="--", alpha=0.6,
           label=f"Best epoch ({best_epoch})")
ax.set_title("LSTM with Early Stopping")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the early-stopped model
y_pred_es_s = model_es.predict(X_test, verbose=0).flatten()
y_pred_es = scaler.inverse_transform(y_pred_es_s.reshape(-1, 1)).flatten()

mae_es = mean_absolute_error(y_actual, y_pred_es)
rmse_es = np.sqrt(mean_squared_error(y_actual, y_pred_es))

print(f"LSTM with Early Stopping -- Test Metrics")
print(f"  MAE:  {mae_es:.2f}")
print(f"  RMSE: {rmse_es:.2f}")

---
## 10. Recursive Multi-Step Forecasting

So far we have done **one-step** forecasting: predict the next value
given a window of known past values.  For **multi-step** forecasting,
we use the model's own predictions as input for subsequent steps.

This is called **recursive** (or iterated) forecasting:

1. Predict $\hat{y}_{t+1}$ from $[y_{t-w+1}, \ldots, y_t]$
2. Predict $\hat{y}_{t+2}$ from $[y_{t-w+2}, \ldots, y_t, \hat{y}_{t+1}]$
3. Continue until the desired horizon

**Warning**: errors accumulate at each step, so recursive forecasts
degrade rapidly over longer horizons.

In [ ]:
def recursive_forecast(model, initial_window, n_steps, scaler):
    """Generate multi-step forecasts recursively.

    Parameters
    ----------
    model : keras Model
        Trained one-step-ahead model.
    initial_window : ndarray, shape (window_size,)
        The last window of scaled observations.
    n_steps : int
        Number of future steps to forecast.
    scaler : MinMaxScaler
        Fitted scaler for inverse transformation.

    Returns
    -------
    forecasts : ndarray, shape (n_steps,)
        Forecasts in original scale.
    """
    window = initial_window.copy()
    forecasts = []

    for _ in range(n_steps):
        # Reshape for LSTM: (1, window_size, 1)
        x_input = window.reshape(1, -1, 1)
        pred = model.predict(x_input, verbose=0)[0, 0]
        forecasts.append(pred)

        # Slide window: drop first, append prediction
        window = np.append(window[1:], pred)

    forecasts = np.array(forecasts)
    return scaler.inverse_transform(forecasts.reshape(-1, 1)).flatten()

In [ ]:
# Use the early-stopped model to make recursive 24-step forecasts
initial_window = scaled[-(n_test + WINDOW):-n_test]  # last window before test period

recursive_preds = recursive_forecast(model_es, initial_window, n_test, scaler)

mae_recursive = mean_absolute_error(y_actual, recursive_preds)
rmse_recursive = np.sqrt(mean_squared_error(y_actual, recursive_preds))

print(f"Recursive 24-step Forecast")
print(f"  MAE:  {mae_recursive:.2f}")
print(f"  RMSE: {rmse_recursive:.2f}")

In [ ]:
# Compare one-step vs recursive forecasts
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(airline.index, airline["Passengers"], color="tab:blue", alpha=0.4, label="Full series")
ax.plot(test_dates, y_actual, color="tab:green", marker="o", markersize=4, label="Test actual")
ax.plot(test_dates, y_pred_es, color="tab:red", linestyle="--",
        label=f"One-step (MAE={mae_es:.1f})")
ax.plot(test_dates, recursive_preds, color="tab:purple", linestyle="-.",
        label=f"Recursive (MAE={mae_recursive:.1f})")
ax.axvline(test_dates[0], color="grey", linestyle=":", alpha=0.6)
ax.set_title("One-Step vs Recursive Multi-Step Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Recursive forecasts degrade over time as errors compound.")
print("Seq2seq models (notebook 03) address this by predicting all steps at once.")

---
## 11. Model Comparison

In [ ]:
# Summary table
comparison = pd.DataFrame([
    {"Model": "Simple LSTM", "MAE": mae_lstm, "RMSE": rmse_lstm},
    {"Model": "Stacked LSTM", "MAE": mae_stacked, "RMSE": rmse_stacked},
    {"Model": "Bidirectional LSTM", "MAE": mae_bidir, "RMSE": rmse_bidir},
    {"Model": "Stacked + Early Stop", "MAE": mae_es, "RMSE": rmse_es},
    {"Model": "Recursive (24-step)", "MAE": mae_recursive, "RMSE": rmse_recursive},
])

print(comparison.to_string(index=False, float_format="{:.2f}".format))
print()
print("One-step forecasts have an unfair advantage (they use true values as input).")
print("Recursive forecasts are the realistic scenario for multi-step ahead prediction.")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(comparison["Model"], comparison["MAE"], color="tab:blue", alpha=0.7,
               edgecolor="black")
ax.set_xlabel("Test MAE (Thousands of Passengers)")
ax.set_title("LSTM Variant Comparison -- Airline Passengers")
ax.grid(True, alpha=0.3, axis="x")
for bar, val in zip(bars, comparison["MAE"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}", va="center")
plt.tight_layout()
plt.show()

---
## Summary

- **Simple RNNs** process sequences step-by-step with shared weights and a hidden
  state: $h_t = \tanh(W_{hh}h_{t-1} + W_{xh}x_t + b)$

- The **vanishing gradient problem** prevents simple RNNs from learning
  long-range dependencies: gradients decay as $\|W\|^T$

- **LSTMs** solve this with a cell state (additive updates) and three
  gates (forget, input, output) that control information flow

- **GRUs** simplify LSTMs to two gates (update, reset) with comparable
  performance and fewer parameters

- **Stacked LSTMs** use `return_sequences=True` on intermediate layers
  to build deeper models

- **Bidirectional LSTMs** process the sequence in both directions,
  giving each step access to past and future context within the window

- **Early stopping** prevents overfitting by monitoring validation loss

- **Recursive forecasting** uses the model's own predictions as input
  for multi-step forecasts, but errors accumulate

- Seq2seq models (next notebook) generate multi-step forecasts in a
  single forward pass, avoiding error accumulation

In [ ]:
print("Next notebook: Sequence-to-sequence models for multi-step forecasting.")